# XGBoost on Summary Statistics — Gradient Boosting for Exoplanet Detection

## Hypothesis

Gradient Boosting (XGBoost) builds trees sequentially, each correcting the residuals of the previous. Unlike Random Forest (independent trees), this allows it to focus on hard cases — stars where RV scatter is moderate but activity correlation is strong.

We test two configurations: (1) 16 physical features (same as RF), (2) 21 features (16+5 enhanced activity).

We track train/val/test ROC-AUC and PR-AUC at every boosting round to detect overfitting.

Hypothesis: XGBoost will use the features more efficiently than RF and reach 0.80-0.83 AUC.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix
import matplotlib.pyplot as plt
import random

# Seed everything
seed = 42
random.seed(seed)
np.random.seed(seed)
xgb.set_config(verbosity=0)

# Load data
observations = pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl')

# --- Compute 16 physical features per star ---
def compute_physical_features(group):
    features = {}
    rv = group['rv_centered']
    rhkp = group['RHKp']
    halpha = group['Halpha']
    rv_err = group['rv_err']

    features['rv_std'] = rv.std()
    features['rv_range'] = rv.max() - rv.min()
    features['rv_mean_abs_dev'] = (rv - rv.median()).abs().mean()
    features['rv_skew'] = rv.skew()
    features['rv_kurtosis'] = rv.kurtosis()
    features['rv_err_mean'] = rv_err.mean()
    features['rv_err_std'] = rv_err.std()
    features['rhkp_std'] = rhkp.std()
    features['rhkp_range'] = rhkp.max() - rhkp.min()
    features['rhkp_mean'] = rhkp.mean()
    features['halpha_std'] = halpha.std()
    features['halpha_range'] = halpha.max() - halpha.min()
    features['halpha_mean'] = halpha.mean()
    features['rv_rhkp_corr'] = rv.corr(rhkp)
    features['rv_halpha_corr'] = rv.corr(halpha)
    features['rhkp_halpha_corr'] = rhkp.corr(halpha)
    features['has_exoplanets'] = group['has_exoplanets'].iloc[0]
    return pd.Series(features)

# --- Compute 5 enhanced activity features per star ---
def compute_enhanced_features(group):
    features = {}
    rv = group['rv_centered'].values
    rhkp = group['RHKp'].values
    halpha = group['Halpha'].values
    bjd = group['bjd'].values
    rv_std = np.std(rv)

    # Absolute correlations (guard BOTH variables to avoid NaN / 0-std divide warnings)
    features['rv_rhkp_corr_abs'] = abs(np.corrcoef(rv, rhkp)[0, 1]) if rv_std > 0 and np.std(rhkp) > 0 else 0.0
    features['rv_halpha_corr_abs'] = abs(np.corrcoef(rv, halpha)[0, 1]) if rv_std > 0 and np.std(halpha) > 0 else 0.0

    # Partial correlation (remove linear time trend from rv and rhkp)
    if len(bjd) > 2 and rv_std > 0 and np.std(rhkp) > 0:
        t = bjd - bjd.mean()
        A = np.vstack([t, np.ones(len(t))]).T
        rv_coef = np.linalg.lstsq(A, rv, rcond=None)[0]
        rv_r = rv - (rv_coef[0] * t + rv_coef[1])
        rhkp_coef = np.linalg.lstsq(A, rhkp, rcond=None)[0]
        rhkp_r = rhkp - (rhkp_coef[0] * t + rhkp_coef[1])
        if np.std(rv_r) > 0 and np.std(rhkp_r) > 0:
            features['rv_rhkp_partial_corr'] = np.corrcoef(rv_r, rhkp_r)[0, 1]
        else:
            features['rv_rhkp_partial_corr'] = 0.0
    else:
        features['rv_rhkp_partial_corr'] = 0.0

    # Rolling-window correlation std
    w = min(20, len(rv) // 2)
    if w >= 5 and rv_std > 0 and np.std(rhkp) > 0:
        corrs = []
        for s in range(0, len(rv) - w + 1, max(1, w // 2)):
            if np.std(rhkp[s:s + w]) > 0 and np.std(rv[s:s + w]) > 0:
                c = np.corrcoef(rv[s:s + w], rhkp[s:s + w])[0, 1]
                if not np.isnan(c):
                    corrs.append(c)
        features['rv_rhkp_corr_std'] = np.std(corrs) if len(corrs) >= 2 else 0.0
    else:
        features['rv_rhkp_corr_std'] = 0.0

    # Ratio of activity variance to RV variance
    features['rhkp_rv_ratio'] = np.std(rhkp) / rv_std if rv_std > 0 else 0.0
    return pd.Series(features)

# Compute features per star. Select only the needed columns (excluding the
# grouping key) so the group handed to apply does not include star_name; this
# avoids the pandas DeprecationWarning about grouping columns in apply.
phys_cols = ['rv_centered', 'RHKp', 'Halpha', 'rv_err', 'has_exoplanets']
enh_cols = ['rv_centered', 'RHKp', 'Halpha', 'bjd']
physical_df = observations.groupby('star_name')[phys_cols].apply(compute_physical_features).reset_index()
enhanced_df = observations.groupby('star_name')[enh_cols].apply(compute_enhanced_features).reset_index()

# Merge (1:1 on star_name); NaNs from undefined correlations -> 0
df = physical_df.merge(enhanced_df, on='star_name', how='left').fillna(0)

# Define feature sets
physical_features = [
    'rv_std', 'rv_range', 'rv_mean_abs_dev', 'rv_skew', 'rv_kurtosis',
    'rv_err_mean', 'rv_err_std',
    'rhkp_std', 'rhkp_range', 'rhkp_mean',
    'halpha_std', 'halpha_range', 'halpha_mean',
    'rv_rhkp_corr', 'rv_halpha_corr', 'rhkp_halpha_corr'
]
enhanced_features_list = [
    'rv_rhkp_corr_abs', 'rv_halpha_corr_abs', 'rv_rhkp_partial_corr',
    'rv_rhkp_corr_std', 'rhkp_rv_ratio'
]
all_features = physical_features + enhanced_features_list

print(f'16 physical features: {len(physical_features)}')
print(f'5 enhanced features: {len(enhanced_features_list)}')
print(f'21 total features: {len(all_features)}')

# Labels and feature matrices
y = df['has_exoplanets'].values.astype(int)
X_16 = df[physical_features].values
X_21 = df[all_features].values

# Stratified 60/20/20 split. A single permutation of indices is shared by both
# feature sets so the same stars land in the same split for both models.
idx_all = np.arange(len(y))
idx_train, idx_temp = train_test_split(idx_all, test_size=0.4, random_state=seed, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=seed, stratify=y[idx_temp])

X_train_16, X_val_16, X_test_16 = X_16[idx_train], X_16[idx_val], X_16[idx_test]
X_train_21, X_val_21, X_test_21 = X_21[idx_train], X_21[idx_val], X_21[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

# Standardize using training statistics only (no leakage from val/test).
# NOTE: XGBoost is a tree model and is invariant to affine rescaling, so this
# does not change the model; kept for consistency with the RF pipeline and any
# future distance-/linear-based baselines.
mean_16, std_16 = X_train_16.mean(axis=0), np.clip(X_train_16.std(axis=0), 1e-8, None)
X_train_16 = (X_train_16 - mean_16) / std_16
X_val_16 = (X_val_16 - mean_16) / std_16
X_test_16 = (X_test_16 - mean_16) / std_16

mean_21, std_21 = X_train_21.mean(axis=0), np.clip(X_train_21.std(axis=0), 1e-8, None)
X_train_21 = (X_train_21 - mean_21) / std_21
X_val_21 = (X_val_21 - mean_21) / std_21
X_test_21 = (X_test_21 - mean_21) / std_21

print(f'Splits: train={len(y_train)} ({y_train.sum()} pos), val={len(y_val)} ({y_val.sum()} pos), test={len(y_test)} ({y_test.sum()} pos)')

In [ ]:
# Parameters shared by both models.
# scale_pos_weight = sqrt(neg/pos): a common heuristic for imbalanced binaries
# that nudges toward the minority class without fully equalizing it.
params = {
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'aucpr'],
    'tree_method': 'hist',
    'scale_pos_weight': np.sqrt((len(y_train) - y_train.sum()) / y_train.sum()),
    'n_estimators': 300,
    'max_depth': 5,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'reg_alpha': 0.01,
    'reg_lambda': 5,
    'random_state': seed,
    'verbosity': 0,
    'early_stopping_rounds': 20
}

# IMPORTANT: XGBoost uses the LAST dataset in eval_set for early stopping
# (i.e. for model selection). Put validation LAST so early stopping is driven
# by validation only. The test set is included solely so its per-round metric is
# recorded for the learning-curve plot; it is never used to choose the model.
# Using the test set for early stopping would leak the test set.
# eval_set order -> validation_0: train, validation_1: test, validation_2: val
eval_16 = [(X_train_16, y_train), (X_test_16, y_test), (X_val_16, y_val)]
eval_21 = [(X_train_21, y_train), (X_test_21, y_test), (X_val_21, y_val)]

print('Training XGBoost on 16 features...')
model_16 = xgb.XGBClassifier(**params)
model_16.fit(X_train_16, y_train, eval_set=eval_16, verbose=False)

print('Training XGBoost on 21 features...')
model_21 = xgb.XGBClassifier(**params)
model_21.fit(X_train_21, y_train, eval_set=eval_21, verbose=False)

model_16.save_model('/kaggle/working/xgb_16features.json')
model_21.save_model('/kaggle/working/xgb_21features.json')
print('Models saved.')

In [ ]:
# Learning curves: ROC-AUC and PR-AUC at every boosting round.
# eval_set order: validation_0=train, validation_1=test, validation_2=val.
results_16 = model_16.evals_result()
results_21 = model_21.evals_result()

n_rounds_16 = len(results_16['validation_0']['auc'])
n_rounds_21 = len(results_21['validation_0']['auc'])


def plot_curve(ax, results, n_rounds, metric, title, ylabel):
    r = range(1, n_rounds + 1)
    ax.plot(r, results['validation_0'][metric], label='Train')
    ax.plot(r, results['validation_2'][metric], label='Val')
    ax.plot(r, results['validation_1'][metric], label='Test')
    ax.set_title(title)
    ax.set_xlabel('Boosting Round')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_curve(axes[0, 0], results_16, n_rounds_16, 'auc',   'XGBoost 16 features — ROC-AUC', 'ROC-AUC')
plot_curve(axes[0, 1], results_21, n_rounds_21, 'auc',   'XGBoost 21 features — ROC-AUC', 'ROC-AUC')
plot_curve(axes[1, 0], results_16, n_rounds_16, 'aucpr', 'XGBoost 16 features — PR-AUC',  'PR-AUC')
plot_curve(axes[1, 1], results_21, n_rounds_21, 'aucpr', 'XGBoost 21 features — PR-AUC',  'PR-AUC')

plt.tight_layout()
plt.show()

In [ ]:
# Predictions. With early stopping, the model uses best_iteration
# (chosen on the validation set) automatically when predicting.
y_val_pred_16 = model_16.predict_proba(X_val_16)[:, 1]
y_val_pred_21 = model_21.predict_proba(X_val_21)[:, 1]
y_pred_16 = model_16.predict_proba(X_test_16)[:, 1]
y_pred_21 = model_21.predict_proba(X_test_21)[:, 1]

# Threshold-free metrics on the TEST set
roc_auc_16 = roc_auc_score(y_test, y_pred_16)
roc_auc_21 = roc_auc_score(y_test, y_pred_21)
pr_auc_16 = average_precision_score(y_test, y_pred_16)
pr_auc_21 = average_precision_score(y_test, y_pred_21)


# Choose the decision threshold on the VALIDATION set (maximize F1), then apply
# that fixed threshold to the test set. Selecting the threshold on the test set
# (the previous approach) tunes on the evaluation data and biases the reported F1.
def best_f1_threshold(y_true, scores):
    prec, rec, thr = precision_recall_curve(y_true, scores)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    idx = int(np.argmax(f1))
    return (float(thr[idx]) if idx < len(thr) else 0.5), float(f1[idx])


best_thresh_16, val_f1_16 = best_f1_threshold(y_val, y_val_pred_16)
best_thresh_21, val_f1_21 = best_f1_threshold(y_val, y_val_pred_21)

# Apply the validation-chosen threshold to the test set
preds_16 = (y_pred_16 >= best_thresh_16).astype(int)
preds_21 = (y_pred_21 >= best_thresh_21).astype(int)
cm_16 = confusion_matrix(y_test, preds_16, labels=[0, 1])
cm_21 = confusion_matrix(y_test, preds_21, labels=[0, 1])


# Honest test-set F1 at the validation-chosen threshold
def f1_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0


test_f1_16 = f1_from_cm(cm_16)
test_f1_21 = f1_from_cm(cm_21)

print(f'XGBoost 16 features: ROC-AUC={roc_auc_16:.4f}, PR-AUC={pr_auc_16:.4f}, '
      f'Val F1={val_f1_16:.4f}, Test F1={test_f1_16:.4f} (thresh={best_thresh_16:.4f})')
print(f'XGBoost 21 features: ROC-AUC={roc_auc_21:.4f}, PR-AUC={pr_auc_21:.4f}, '
      f'Val F1={val_f1_21:.4f}, Test F1={test_f1_21:.4f} (thresh={best_thresh_21:.4f})')

print(f'\nConfusion Matrix (16 features, threshold={best_thresh_16:.3f}):')
print(cm_16)
print(f'Confusion Matrix (21 features, threshold={best_thresh_21:.3f}):')
print(cm_21)

In [ ]:
# Feature importances (gain-based)
importances_16 = model_16.feature_importances_
importances_21 = model_21.feature_importances_

# Sort and print
imp_df_16 = pd.DataFrame({'feature': physical_features, 'importance': importances_16})
imp_df_16 = imp_df_16.sort_values('importance', ascending=False)

imp_df_21 = pd.DataFrame({'feature': all_features, 'importance': importances_21})
imp_df_21 = imp_df_21.sort_values('importance', ascending=False)

print(f'Top 10 Feature Importances (16 features):')
for _, row in imp_df_16.head(10).iterrows():
    print(f'  {row["feature"]:<25s} {row["importance"]:.4f}')

print(f'Top 10 Feature Importances (21 features):')
for _, row in imp_df_21.head(10).iterrows():
    marker = ' *NEW' if row["feature"] in enhanced_features_list else ''
    print(f'  {row["feature"]:<25s} {row["importance"]:.4f}{marker}')


In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {"Model": "RF (16 physical features)",    "Input": "per-star aggregates",     "ROC-AUC": 0.7994, "PR-AUC": 0.4658, "Best F1": 0.5221},
    {"Model": "Transformer (raw sequences)",  "Input": "(T, 21) raw obs",          "ROC-AUC": 0.6793, "PR-AUC": float('nan'), "Best F1": float('nan')},
    {"Model": "CNN + pos encoding (21 ch)",   "Input": "(T, 21) raw obs",          "ROC-AUC": 0.6490, "PR-AUC": 0.3049, "Best F1": 0.3887},
    {"Model": "CNN stripped (4 ch)",           "Input": "(T, 4) raw obs",           "ROC-AUC": 0.6391, "PR-AUC": 0.2897, "Best F1": 0.3855},
    {"Model": "Hybrid RF+CNN",                 "Input": "16 stats + (T,4) raw",      "ROC-AUC": 0.6932, "PR-AUC": 0.3359, "Best F1": 0.4091},
    {"Model": "XGBoost (16 features)",        "Input": "16 physical features",      "ROC-AUC": roc_auc_16, "PR-AUC": pr_auc_16, "Best F1": test_f1_16},
    {"Model": "XGBoost (21 features)",        "Input": "16+5 enhanced features",   "ROC-AUC": roc_auc_21, "PR-AUC": pr_auc_21, "Best F1": test_f1_21},
])

# "Best F1" for XGBoost is the test-set F1 at a threshold selected on the
# VALIDATION set (an honest estimate). Values for the other models are taken
# from their own notebooks and may use a different threshold-selection protocol.
print(comparison.to_string(index=False, float_format=lambda x: f'{x:.4f}' if not pd.isna(x) else 'NaN'))

print()
if roc_auc_16 > 0.7994:
    print(f'✓ XGBoost (16) ({roc_auc_16:.4f}) BEATS RF baseline (0.7994).')
elif roc_auc_16 > 0.6932:
    print(f'• XGBoost (16) ({roc_auc_16:.4f}) beats all DL models but trails RF (0.7994).')
else:
    print(f'✗ XGBoost (16) ({roc_auc_16:.4f}) trails RF baseline (0.7994).')